# To further reproduce this paper, we should refer to most likely used dataset at that time
- https://www.tycho.pitt.edu/data/level3.php?utm_source=chatgpt.com, LEVEL 2 Data, version 1.1.0
- Article: https://academic.oup.com/ofid/article/5/7/ofy137/5039595

In [12]:
core_columns = [
    "disease",
    "state",
    "loc_type",
    "from_date",
    "number",
    " event",
]

In [ ]:
import pandas as pd

df = pd.read_csv('../raw/tycho/ProjectTycho_Level2_v1.1.0.csv', low_memory=False)
df.head()

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url
0,188824,US,PA,PHILADELPHIA,CITY,TYPHOID FEVER [ENTERIC FEVER],DEATHS,14,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...
1,188824,US,PA,PHILADELPHIA,CITY,SCARLET FEVER,DEATHS,4,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...
2,188824,US,PA,PHILADELPHIA,CITY,DIPHTHERIA,DEATHS,4,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...
3,188826,US,PA,PHILADELPHIA,CITY,TYPHOID FEVER [ENTERIC FEVER],DEATHS,12,1888-06-24,1888-06-30,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...
4,188826,US,PA,PHILADELPHIA,CITY,SCARLET FEVER,DEATHS,5,1888-06-24,1888-06-30,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...


In [14]:
df['Year'] = df['from_date'].str.split('-').str[0].astype(int)
df.head()

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url,Year
0,188824,US,PA,PHILADELPHIA,CITY,TYPHOID FEVER [ENTERIC FEVER],DEATHS,14,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...,1888
1,188824,US,PA,PHILADELPHIA,CITY,SCARLET FEVER,DEATHS,4,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...,1888
2,188824,US,PA,PHILADELPHIA,CITY,DIPHTHERIA,DEATHS,4,1888-06-10,1888-06-16,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...,1888
3,188826,US,PA,PHILADELPHIA,CITY,TYPHOID FEVER [ENTERIC FEVER],DEATHS,12,1888-06-24,1888-06-30,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...,1888
4,188826,US,PA,PHILADELPHIA,CITY,SCARLET FEVER,DEATHS,5,1888-06-24,1888-06-30,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC20...,1888


In [15]:
df['Year'].min(), df['Year'].max()

(np.int64(1887), np.int64(2014))

### Recal filters done by article
- state level only, NOT city
- disease: measles
- cases only, NOT deaths


In [16]:
exclude_regions = [
    'AS',  # American Samoa
    'GU',  # Guam
    'MP',  # Northern Mariana Islands
    'PR',  # Puerto Rico
    'VI',  # U.S. Virgin Islands
    'HI',
    'AK',
    'PT',
]

measles_df = df[(df["disease"] == "MEASLES" )
                & (df['loc_type'] == "STATE")
                & (df[' event'] == "CASES")
                & (~df['state'].isin(exclude_regions))
                & (df['Year'] >= 1931) 
                & (df['Year'] <= 1992) 
                ]

measles_df['from_date'] = pd.to_datetime(measles_df['from_date'])
measles_df.head()

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url,Year
3206926,193101,US,ME,MAINE,STATE,MEASLES,CASES,7,1931-01-04,1931-01-10,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC19...,1931
3206927,193101,US,NH,NEW HAMPSHIRE,STATE,MEASLES,CASES,21,1931-01-04,1931-01-10,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC19...,1931
3206928,193101,US,VT,VERMONT,STATE,MEASLES,CASES,14,1931-01-04,1931-01-10,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC19...,1931
3206929,193101,US,MA,MASSACHUSETTS,STATE,MEASLES,CASES,630,1931-01-04,1931-01-10,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC19...,1931
3206930,193101,US,RI,RHODE ISLAND,STATE,MEASLES,CASES,1,1931-01-04,1931-01-10,http://www.ncbi.nlm.nih.gov/pmc/articles/PMC19...,1931


In [17]:
print("States:", measles_df['state'].nunique())
print("Rows:", len(measles_df))
print("Start:", measles_df['from_date'].min())
print("End:", measles_df['from_date'].max())

print(measles_df['loc_type'].value_counts())
print(measles_df[' event'].value_counts())
print(measles_df['disease'].value_counts())

States: 49
Rows: 102972
Start: 1931-01-04 00:00:00
End: 1992-12-20 00:00:00
loc_type
STATE    102972
Name: count, dtype: int64
 event
CASES    102972
Name: count, dtype: int64
disease
MEASLES    102972
Name: count, dtype: int64


In [18]:
dupes = measles_df[
    measles_df.duplicated(
        subset=['state', 'from_date'],
        keep=False
    )
].sort_values(['state', 'from_date'])

dupes.head(50)

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url,Year
3293791,194107,US,AL,ALABAMA,STATE,MEASLES,CASES,140,1941-02-09,1941-02-15,https://www.tycho.pitt.edu/raw/PDF/1941/08.pdf,1941
3293972,194107,US,AL,ALABAMA,STATE,MEASLES,CASES,140,1941-02-09,1941-02-15,https://www.tycho.pitt.edu/raw/PDF/1941/09.pdf,1941
3364720,195506,US,AL,ALABAMA,STATE,MEASLES,CASES,30,1955-02-06,1955-02-12,http://stacks.cdc.gov/view/cdc/694/,1955
3364770,195506,US,AL,ALABAMA,STATE,MEASLES,CASES,94,1955-02-06,1955-02-12,http://stacks.cdc.gov/view/cdc/695/,1955
3366812,195549,US,AL,ALABAMA,STATE,MEASLES,CASES,9,1955-12-04,1955-12-10,http://stacks.cdc.gov/view/cdc/787/,1955
3366860,195549,US,AL,ALABAMA,STATE,MEASLES,CASES,6,1955-12-04,1955-12-10,http://stacks.cdc.gov/view/cdc/790/,1955
3370381,195718,US,AL,ALABAMA,STATE,MEASLES,CASES,400,1957-04-28,1957-05-04,http://stacks.cdc.gov/view/cdc/944/,1957
3370433,195718,US,AL,ALABAMA,STATE,MEASLES,CASES,456,1957-04-28,1957-05-04,http://stacks.cdc.gov/view/cdc/945/,1957
3399978,197107,US,AL,ALABAMA,STATE,MEASLES,CASES,22,1971-02-14,1971-02-20,http://stacks.cdc.gov/view/cdc/1736/,1971
3400020,197107,US,AL,ALABAMA,STATE,MEASLES,CASES,46,1971-02-14,1971-02-20,http://stacks.cdc.gov/view/cdc/1737/,1971


In [19]:
dupes[
    (dupes['state'] == 'PA') &
    (dupes['from_date'] == '1935-01-06')
]

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url,Year


In [20]:
dupe_summary = (
    dupes
    .groupby(['state', 'from_date'])
    .agg(
        rows=('number', 'size'),
        unique_counts=('number', 'nunique'),
        min_count=('number', 'min'),
        max_count=('number', 'max')
    )
    .reset_index()
)

dupe_summary.head()

,state,from_date,rows,unique_counts,min_count,max_count
0,AL,1941-02-09,2,1,140,140
1,AL,1955-02-06,2,2,30,94
2,AL,1955-12-04,2,2,6,9
3,AL,1957-04-28,2,2,400,456
4,AL,1971-02-14,2,2,22,46


In [21]:
dupe_summary['unique_counts'].value_counts()

unique_counts
2    1104
1     110
4       3
3       1
Name: count, dtype: int64

In [23]:
conflict = dupe_summary[
    dupe_summary['unique_counts'] > 1
].iloc[0]

measles_df[
    (measles_df['state'] == conflict['state']) &
    (measles_df['from_date'] == conflict['from_date'])
]

,epi_week,country,state,loc,loc_type,disease,event,number,from_date,to_date,url,Year
3364720,195506,US,AL,ALABAMA,STATE,MEASLES,CASES,30,1955-02-06,1955-02-12,http://stacks.cdc.gov/view/cdc/694/,1955
3364770,195506,US,AL,ALABAMA,STATE,MEASLES,CASES,94,1955-02-06,1955-02-12,http://stacks.cdc.gov/view/cdc/695/,1955


### It looks like urls are different thus casing dupes. We will no longer analyze this and instead focus on dataset version 1.0

# Data imputation begins here

In [22]:
assert False

AssertionError: 

In [ ]:
essential_cols = ['state', 'from_date', 'number', 'Year'] # select columns to make imputation easier
measles_df = measles_df[essential_cols]

In [ ]:
missing_dates = pd.DataFrame(columns=essential_cols)
for state in measles_df['state'].unique(): # iterate through reproducible dataset for each state

    state_df = measles_df[measles_df['state'] == state].sort_values(by='from_date')

    for i in range(len(state_df)):
        current_row = state_df.iloc[i]

        # Next row
        if i + 1 < len(state_df):
            next_row = state_df.iloc[i + 1]
        else:
            next_row = None

        # we only start imputation if current row is not the first or the last
        if next_row is not None:
            
            # first we check that the next row is the following week from the current row
            day_difference = next_row["from_date"] - current_row["from_date"]
            if day_difference.days > 7:

                # generate the missing dates
                new_dates = pd.date_range(
                    start=current_row["from_date"] + pd.Timedelta(days=7),
                    end=next_row["from_date"] - pd.Timedelta(days=7),
                    freq="7D"
                )

                count_val = (current_row['number'] + next_row['number']) / 2
                for date in new_dates:

                    row_dict = {
                        'from_date': pd.Timestamp(date),
                        'state':current_row['state'],
                        'number': count_val,
                        'Year': date.year
                        }
                    
                    pd_series = pd.Series(row_dict)
                    missing_dates = pd.concat([missing_dates, pd_series.to_frame().T], ignore_index=True)

In [ ]:
measles_df = pd.concat([measles_df, missing_dates], ignore_index=True)

## Let's reproduce this paper: https://academic.oup.com/ofid/article/5/7/ofy137/5039595
Here's Table 1 as a markdown table:

**Table 1. Observed and Prevented Measles Cases, Deaths, and Related Costs in the United States, With 80% Uncertainty Range**

| | Prevaccination (1931–1963) | Introduction (1964–1970) | 1-dose Vaccine (1971–1989) | 2-dose Vaccine (1990–2014) | Entire Period (1964–2014) |
|---|---|---|---|---|---|
| **Cases, millions** | | | | | |
| Observed<sup>a</sup> | 16.81 | 1.14 | 0.39 | 0.05 | 1.57 |
| Prevented | — | 2.49 (0.30 to 8.35) | 10.12 (3.19 to 31.89) | 17.17 (5.59 to 57.60) | 29.78 (9.08 to 97.84) |
| **Deaths, thousands** | | | | | |
| Observed | 45.52 | 1.19 | 0.28 | 0.11 | 1.59 |
| Prevented | — | 1.46 (–1.18 to 10.50) | 10.51 (–0.28 to 76.48) | 19.61 (–0.11 to 334.61) | 31.57 (–1.57 to 421.59) |

<sup>a</sup>Observed cases, as reported by the US Centers for Disease Control and Prevention (CDC; data from Project Tycho and the CDC).
<sup>b</sup>Estimated costs due to hospitalization or lost income associated with reported measles cases based on state-level cost estimates.

### ***Note that project Tycho is between 1931 and 1992***
- Alaska and Hawaii are excluded
- Not coutning sub-state categories
- 32% weekly missing values - current dataset is pre-imuptation
- Only consider incident counts: PartOfCumulativeCountSeries==0

In [ ]:
len(missing_dates) / len(measles_df)

0.33174119021351156

In [ ]:
prevaccination = measles_df[(measles_df['Year'] >= 1931) & (measles_df['Year'] <= 1963)]
print(prevaccination['number'].sum())

16832929.0


In [ ]:
introduction = measles_df[(measles_df['Year'] >= 1964) & (measles_df['Year'] <= 1970)]
print(introduction['number'].sum())

1093317.5


In [ ]:
one_dose_vaccine = measles_df[(measles_df['Year'] >= 1971) & (measles_df['Year'] <= 1989)]
print(round(one_dose_vaccine['number'].sum(), 2))

570158.5
